# 32 — Context Windows, Masks, Packing, and Storage

**Master syllabus coverage:** V2 4.10–4.12

## Why this matters

Correct token IDs can still create corrupt training data through off-by-one targets, future leakage, padding loss, or document boundary leakage. Sequence construction defines the actual learning task.

## Learning objectives

- Create aligned next-token inputs and targets.
- Distinguish causal, padding, document, and loss masks.
- Pack documents with an explicit cross-document attention policy.
- Window and store token data with boundary and provenance metadata.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Shift a token sequence into inputs and targets

Given tokens $[x_0,x_1,\ldots,x_T]$, inputs are $[x_0,\ldots,x_{T-1}]$ and targets
are $[x_1,\ldots,x_T]$. The model at position $t$ predicts target $x_{t+1}$. A context
window of length $T$ therefore requires $T+1$ source tokens to make $T$ training pairs.


In [ ]:
import numpy as np
import torch

PAD, BOS, EOS = 0, 1, 2
document = [BOS, 10, 11, 12, EOS]
inputs = torch.tensor(document[:-1])
targets = torch.tensor(document[1:])
print("inputs: ", inputs.tolist())
print("targets:", targets.tolist())
assert torch.equal(inputs[1:], targets[:-1])


## 2. Causal and padding masks answer different questions

A causal mask prevents query position $t$ from reading key positions $>t$. A padding
mask prevents attention to nonexistent padded keys. A loss mask decides which target
positions contribute to the objective. They have different shapes and must not be
confused.

Causal allowed mask: `allowed[t_query, t_key] = (t_key <= t_query)` with `[T,T]`.


In [ ]:
T = 5
causal_allowed = torch.tril(torch.ones(T, T, dtype=torch.bool))  # [Tq,Tk]
padding_valid = torch.tensor([[True, True, True, True, False],
                              [True, True, True, False, False]])  # [B,Tk]
combined_allowed = causal_allowed[None, :, :] & padding_valid[:, None, :]  # [B,Tq,Tk]

print("causal:\n", causal_allowed.int())
print("combined batch 1:\n", combined_allowed[1].int())
assert combined_allowed.shape == (2, T, T)
assert not combined_allowed[:, 0, 1:].any()


## 3. Pack documents without cross-document leakage

Concatenating documents improves utilization, but ordinary causal attention would let
later documents read earlier documents. EOS boundaries may be acceptable for continuous
pretraining when intentionally modeled; strict document isolation requires a block-
diagonal attention mask or resetting sequence segments. The policy must be explicit.


In [ ]:
documents = [[BOS, 10, 11, EOS], [BOS, 20, 21, 22, EOS]]
packed = [token for doc in documents for token in doc]
segment_ids = [segment for segment, doc in enumerate(documents) for _ in doc]
packed_t = torch.tensor(packed)
segments_t = torch.tensor(segment_ids)
length = len(packed)

causal = torch.tril(torch.ones(length, length, dtype=torch.bool))
same_document = segments_t[:, None] == segments_t[None, :]
isolated_attention = causal & same_document
assert not isolated_attention[4:, :4].any()
print("packed IDs:", packed)
print("segment IDs:", segment_ids)


## 4. Fixed windows and boundary-safe labels

For long streams, choose windows by a documented stride. Overlapping windows expose
tokens multiple times; non-overlapping windows are cheaper but waste tails. If windows
are independent, never create a target that belongs outside the retained source boundary.
Pad short windows and set padded targets to an ignore index.


In [ ]:
def make_windows(tokens: list[int], context: int, stride: int, ignore_index: int = -100):
    examples = []
    for start in range(0, max(1, len(tokens) - 1), stride):
        chunk = tokens[start:start + context + 1]
        if len(chunk) < 2:
            break
        x, y = chunk[:-1], chunk[1:]
        valid = len(x)
        x = x + [PAD] * (context - valid)
        y = y + [ignore_index] * (context - valid)
        examples.append((x, y, valid))
        if start + context + 1 >= len(tokens):
            break
    return examples

windows = make_windows(packed, context=5, stride=5)
for x, y, valid in windows:
    print("x=", x, "y=", y, "valid=", valid)
    assert len(x) == len(y) == 5 and all(value == -100 for value in y[valid:])


## 5. Storage needs schema and provenance

Token arrays may use compact integer dtypes when vocabulary size permits. Alongside IDs,
store tokenizer version/hash, normalization policy, document boundaries, split identity,
filtering/deduplication version, and checksums. Memory mapping avoids loading an entire
corpus but does not replace shuffled, boundary-aware sampling.


In [ ]:
vocab_size = 50_000
max_id = vocab_size - 1
dtype = np.uint16 if max_id <= np.iinfo(np.uint16).max else np.uint32
token_array = np.asarray(packed, dtype=dtype)
metadata = {
    "dtype": str(token_array.dtype),
    "token_count": int(token_array.size),
    "vocab_size": vocab_size,
    "tokenizer_version": "lesson-demo-v1",
    "split": "train",
}
print(metadata, "bytes:", token_array.nbytes)
assert token_array.max() < vocab_size


## Failure modes to recognize

- **Unshifted targets:** the model learns to copy the current token.
- **Future-visible attention:** training loss is excellent but autoregressive generation fails.
- **Padding loss:** batch shape artifacts dominate the objective.
- **Cross-document leakage:** unrelated or evaluation content becomes accessible context.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Build a padded batch and produce attention, target, and loss masks with explicit shapes.
2. Compare token utilization for padded, packed, and overlapping-window strategies.
3. Write assertions proving no query can attend to a future or different-document key.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can turn documents into fixed training tensors and prove there is no unintended future, padding, or document leakage.

**Next:** 33 — Embeddings, the residual stream, and position.
